# LeetCode 723 — Candy Crush bằng Python

Link đề: https://leetcode.com/problems/candy-crush/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def candyCrush(self, board: List[List[int]]) -> List[List[int]]:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí trên LeetCode). Đề và các ví dụ trong notebook được viết theo hiểu biết về bài, nên hãy **đối chiếu lại với đề gốc** nếu bạn có tài khoản Premium. Mọi đáp án đều đã được kiểm chứng bằng code (so giữa bốn cách giải rất khác nhau, so với các ca tính tay và với các **tính chất bắt buộc** của bàn cờ ổn định). Các ví dụ ngoài hai ví dụ chính của đề là ví dụ **tự dựng** (đối chiếu lại đề gốc nếu cần).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bàn cờ `m x n` chứa các viên kẹo (số nguyên là loại kẹo, `0` là ô trống). Lặp lại cho đến khi ổn định: **đập đồng thời** mọi dãy `>= 3` kẹo cùng loại liền nhau theo hàng hoặc cột, rồi cho kẹo ở trên **rơi xuống** lấp ô trống; trả về bàn cờ ổn định.
2. Nắm các công cụ Python: danh sách hai chiều `board[i][j]`, sao chép bàn cờ (`[row[:] for row in board]`), **đánh dấu bằng số âm** + `abs`, con trỏ ghi để **dồn kẹo xuống đáy** của mỗi cột, `itertools.groupby` để tìm dãy liên tiếp, `set` các cặp `(i, j)`.
3. Đi từ **mô phỏng "vật lý" chậm (đếm dãy quanh từng ô, kẹo rơi từng ô một)** → **đánh dấu âm tại chỗ + dồn cột bằng con trỏ** → **`groupby` + tập ô bị đập + dựng lại cột** → **chỉ quét lại vùng bị ảnh hưởng sau mỗi lượt rơi**.
4. Hiểu mấu chốt: phải **đánh dấu hết rồi mới đập** (không đập ngay khi thấy) vì hai dãy cắt nhau (chữ L, chữ T, dấu cộng) phải bị đập **cùng lúc**; sau đó mỗi cột chỉ là bài toán **dồn các ô khác 0 xuống đáy, giữ nguyên thứ tự**.
5. Tránh các lỗi kinh điển: đập ngay khi quét làm hỏng dãy cắt nhau, quên `abs` khi so sánh ô đã đánh dấu, chỉ rơi một lượt (quên lặp), rơi sai (một lượt hoán đổi), và quên một trong hai hướng (hàng hoặc cột).

## 2. Đề bài & ví dụ

Đây là bài cài đặt thuật toán "đập kẹo" cơ bản của trò Candy Crush.

Cho mảng số nguyên `board` kích thước `m x n`, `board[i][j]` là **loại** kẹo ở ô `(i, j)`. Giá trị `board[i][j] == 0` nghĩa là ô **trống**.

Bàn cờ đã cho là trạng thái **sau nước đi** của người chơi. Bạn cần đưa bàn cờ về trạng thái **ổn định** theo các quy tắc:

- Nếu có **ba hoặc nhiều hơn** kẹo **cùng loại** nằm liền nhau theo **hàng ngang hoặc cột dọc**, hãy **đập tất cả cùng một lúc**: các ô đó trở thành ô trống.
- Sau khi đập đồng thời, nếu một ô trống có kẹo **ở phía trên**, các kẹo đó **rơi xuống** cho tới khi chạm kẹo khác hoặc chạm đáy (tất cả rơi cùng lúc). **Không** có kẹo mới rơi vào từ mép trên.
- Sau bước trên có thể lại xuất hiện các dãy đập được; nếu có thì lặp lại các bước trên.
- Nếu không còn gì để đập (bàn cờ ổn định), trả về bàn cờ hiện tại.

**Ví dụ 1** (bàn cờ `10 x 5`)

- Input: `board = [[110,5,112,113,114],[210,211,5,213,214],[310,311,3,313,314],[410,411,412,5,414],[5,1,512,3,3],[610,4,1,613,614],[710,1,2,713,714],[810,1,2,1,1],[1,1,2,2,2],[4,1,4,4,1014]]`
- Output: `[[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[110,0,0,0,114],[210,0,0,0,214],[310,0,0,113,314],[410,0,0,213,414],[610,211,112,313,614],[710,311,412,613,714],[810,411,512,713,1014]]`

**Ví dụ 2** (bàn cờ `5 x 5`)

- Input: `board = [[1,3,5,5,2],[3,4,3,3,1],[3,2,4,5,2],[2,4,4,5,5],[1,4,4,1,1]]`
- Output: `[[1,3,0,0,0],[3,4,0,5,2],[3,2,0,3,1],[2,4,0,5,2],[1,4,3,1,1]]`

**Ràng buộc:**

- `m == board.length`, `n == board[i].length`
- `3 <= m, n <= 50`
- `1 <= board[i][j] <= 2000`

Chú ý: đáp án **duy nhất** (quy tắc không có lựa chọn). Cách giải có thể sửa trực tiếp `board` rồi trả về chính nó.

In [1]:
from typing import List, Optional
from collections import Counter
from itertools import groupby
import copy
import random

ex1 = [[110, 5, 112, 113, 114], [210, 211, 5, 213, 214], [310, 311, 3, 313, 314], [410, 411, 412, 5, 414],
       [5, 1, 512, 3, 3], [610, 4, 1, 613, 614], [710, 1, 2, 713, 714], [810, 1, 2, 1, 1], [1, 1, 2, 2, 2],
       [4, 1, 4, 4, 1014]]
ex1_out = [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [110, 0, 0, 0, 114], [210, 0, 0, 0, 214],
           [310, 0, 0, 113, 314], [410, 0, 0, 213, 414], [610, 211, 112, 313, 614], [710, 311, 412, 613, 714],
           [810, 411, 512, 713, 1014]]
ex2 = [[1, 3, 5, 5, 2], [3, 4, 3, 3, 1], [3, 2, 4, 5, 2], [2, 4, 4, 5, 5], [1, 4, 4, 1, 1]]
ex2_out = [[1, 3, 0, 0, 0], [3, 4, 0, 5, 2], [3, 2, 0, 3, 1], [2, 4, 0, 5, 2], [1, 4, 3, 1, 1]]

examples = [(ex1, ex1_out), (ex2, ex2_out)]

# Ví dụ tự dựng (đối chiếu lại đề gốc nếu cần)
cascade_in = [[2, 3, 4], [2, 4, 3], [1, 1, 1], [2, 3, 4]]          # đập hàng 1, cột 2 rơi xuống và lại tạo dãy
cascade_out = [[0, 0, 0], [0, 3, 4], [0, 4, 3], [0, 3, 4]]
cross_in = [[1, 2, 3], [1, 1, 1], [1, 3, 2]]                       # hàng và cột cắt nhau: đập đồng thời
cross_out = [[0, 0, 0], [0, 2, 3], [0, 3, 2]]
stable_in = [[1, 2, 1], [2, 1, 2], [1, 2, 1]]                      # đã ổn định
extra_examples = [
    (cascade_in, cascade_out),
    (cross_in, cross_out),
    (stable_in, [row[:] for row in stable_in]),
]
for b, expected in examples + extra_examples:
    print(len(b), "x", len(b[0]), "-> kỳ vọng có", sum(v == 0 for row in expected for v in row), "ô trống")

10 x 5 -> kỳ vọng có 25 ô trống
5 x 5 -> kỳ vọng có 6 ô trống
4 x 3 -> kỳ vọng có 6 ô trống
3 x 3 -> kỳ vọng có 5 ô trống
3 x 3 -> kỳ vọng có 0 ô trống


## 3. Kiến thức nền

### 3.1. Bàn cờ là danh sách hai chiều; **sao chép** đúng cách

`board[i][j]` là ô ở **hàng `i`** (từ trên xuống), **cột `j`** (từ trái sang). `len(board)` là số hàng `m`, `len(board[0])` là số cột `n`. Phép gán `b = board` **không** sao chép mà chỉ tạo thêm một tên cho cùng một đối tượng; để sao chép **độc lập** ta cần sao chép từng hàng: `[row[:] for row in board]` (hoặc `copy.deepcopy`). Cũng đừng tạo bảng bằng `[[0] * n] * m`: `m` hàng sẽ là **cùng một** list lặp lại.

In [2]:
board = [[1, 2], [3, 4]]
alias = board
clone = [row[:] for row in board]
board[0][0] = 99
print(alias[0][0], clone[0][0])           # 99 1   (alias đổi theo, clone thì không)

bad = [[0] * 2] * 2                        # SAI: hai hàng là cùng một list
bad[0][0] = 7
print(bad)                                 # [[7, 0], [7, 0]]
good = [[0] * 2 for _ in range(2)]         # ĐÚNG: mỗi hàng một list riêng
good[0][0] = 7
print(good)                                # [[7, 0], [0, 0]]

99 1
[[7, 0], [7, 0]]
[[7, 0], [0, 0]]


### 3.2. Đánh dấu bằng **số âm** và `abs`

Loại kẹo luôn `>= 1`, `0` là trống. Vì vậy ta có thể dùng **số âm** để đánh dấu "ô này sẽ bị đập" mà **không mất** thông tin loại kẹo: `-v` vẫn nhắc ta loại là `abs(-v) = v`. Khi so sánh hai ô để tìm dãy, ta so `abs(a) == abs(b)` (vì ô có thể đã bị đánh dấu bởi một dãy khác). Cách này cho phép xử lý các dãy **cắt nhau** mà không cần mảng phụ.

In [3]:
a, b = 5, -5
print(a == b, abs(a) == abs(b))            # False True
print(-abs(a), -abs(b))                    # -5 -5   (đánh dấu: luôn thành số âm, an toàn khi đã âm)
print(abs(0), 0 and abs(0))                # 0 0     (ô trống không bao giờ được coi là kẹo)

False True
-5 -5
0 0


### 3.3. Cho kẹo **rơi xuống**: dồn các ô dương về đáy bằng con trỏ ghi

Với **một cột** (đọc từ trên xuống thành list), "rơi xuống" nghĩa là: các ô **khác trống/chưa bị đập** được giữ nguyên **thứ tự** và dồn về **đáy**, phần trên là `0`. Ta quét từ **dưới lên** với con trỏ ghi `k` (ban đầu ở đáy): gặp ô dương thì ghi vào `k` rồi `k -= 1`; xong thì điền `0` cho mọi vị trí từ `k` lên trên.

In [4]:
col = [3, 0, 4, -2, 5, 0]          # dương: kẹo còn; 0: trống; -2: đã bị đánh dấu (sẽ bị đập)
m = len(col)
k = m - 1
for i in range(m - 1, -1, -1):
    if col[i] > 0:
        col[k] = col[i]
        k -= 1
while k >= 0:
    col[k] = 0
    k -= 1
print(col)                          # [0, 0, 0, 3, 4, 5]

[0, 0, 0, 3, 4, 5]


### 3.4. `itertools.groupby`: tìm các dãy giá trị liên tiếp bằng nhau

`groupby(dãy)` gom các phần tử **liên tiếp bằng nhau** thành nhóm `(giá trị, iterator các phần tử)`. Độ dài nhóm là độ dài dãy liên tiếp. Dùng để tìm "dãy từ 3 trở lên".

In [5]:
row = [1, 1, 1, 2, 3, 3, 3, 3, 0, 0, 0]
pos = 0
for value, grp in groupby(row):
    length = len(list(grp))
    print(f"giá trị {value}: bắt đầu {pos}, dài {length}", "<- dãy >= 3" if value and length >= 3 else "")
    pos += length

giá trị 1: bắt đầu 0, dài 3 <- dãy >= 3
giá trị 2: bắt đầu 3, dài 1 
giá trị 3: bắt đầu 4, dài 4 <- dãy >= 3
giá trị 0: bắt đầu 8, dài 3 


### 3.5. Tập các ô bị đập: `set` các cặp `(i, j)` và chuyển vị `zip(*board)`

Khi cần gom các ô cần đập từ cả hàng và cột, một `set` các cặp `(i, j)` tự loại trùng (ô nằm trên hai dãy cắt nhau chỉ tính một lần). `zip(*board)` cho **các cột** của bàn cờ (chuyển vị).

In [6]:
board = [[1, 2, 3], [4, 5, 6]]
print(list(zip(*board)))                    # [(1, 4), (2, 5), (3, 6)]  -> các cột
marked = set()
marked.update((0, j) for j in range(3))      # một dãy hàng
marked.update((i, 1) for i in range(2))      # một dãy cột, cắt ở (0, 1)
print(sorted(marked), len(marked))           # 4 ô chứ không phải 5

[(1, 4), (2, 5), (3, 6)]
[(0, 0), (0, 1), (0, 2), (1, 1)] 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy bàn cờ `4 x 3` (ví dụ tự dựng "cascade"):

```
2 3 4
2 4 3
1 1 1
2 3 4
```

**Lượt 1 – quét tìm dãy:** hàng 2 (`1 1 1`) là dãy ngang dài `3`. Cột 0 là `2,2,1,2` không có dãy `3` liên tiếp; cột 1, cột 2 không có dãy. Đập hàng 2:

```
2 3 4
2 4 3
. . .      <- ô trống
2 3 4
```

**Rơi xuống:** cột 0: `2,2,.,2` ⇒ `.,2,2,2`; cột 1: `3,4,.,3` ⇒ `.,3,4,3`; cột 2: `4,3,.,4` ⇒ `.,4,3,4`:

```
. . .
2 3 4
2 4 3
2 3 4
```

**Lượt 2:** cột 0 giờ là `.,2,2,2` ⇒ dãy dọc dài `3`! Đập cả cột 0, rơi xuống (không còn gì ở trên). Kết quả:

```
. . .
. 3 4
. 4 3
. 3 4
```

**Lượt 3:** không còn dãy nào ⇒ dừng. Đây là một **phản ứng dây chuyền (cascade)**: sau khi rơi, cấu hình mới có thể tạo dãy mới, nên phải **lặp** cho tới khi một lượt quét **không tìm thấy gì**.

**Vì sao phải đánh dấu hết rồi mới đập?** Xét dấu cộng:

```
1 2 3
1 1 1
1 3 2
```

Có dãy ngang `1 1 1` (hàng 1) và dãy dọc `1 1 1` (cột 0) cắt nhau ở ô `(1, 0)`. Cả **năm** ô phải bị đập cùng lúc. Nếu ta đập hàng trước, rồi rơi xuống, thì cột 0 đã bị phá (`1,.,1`) và ta sẽ **không** đập dãy dọc. Vậy: **quét toàn bàn cờ, đánh dấu mọi ô, rồi mới đập và rơi**.

In [7]:
def show(board, title=None):
    """In bàn cờ; ô trống in dấu chấm, ô đã đánh dấu (âm) in kèm dấu '-'."""
    if title:
        print(title)
    w = max(len(str(abs(v))) for row in board for v in row) + 1
    for row in board:
        print(" ".join(("." if v == 0 else str(v)).rjust(w) for v in row))


show(cascade_in, "cascade ban đầu:")
show(cascade_out, "cascade kết quả (tính tay ở trên):")

cascade ban đầu:
 2  3  4
 2  4  3
 1  1  1
 2  3  4
cascade kết quả (tính tay ở trên):
 .  .  .
 .  3  4
 .  4  3
 .  3  4


## 5. Cách 1 — Brute Force / baseline: mô phỏng "vật lý" chậm

**Ý tưởng.** Làm đúng theo mô tả, không tối ưu gì cả:

1. **Quét:** với **từng ô** có kẹo `v`, **đếm độ dài dãy ngang** đi qua ô đó (đi sang trái và sang phải khi còn gặp `v`) và tương tự **dãy dọc**. Nếu một trong hai `>= 3` thì ô này bị đánh dấu.
2. **Đập:** đặt mọi ô bị đánh dấu thành `0` (sau khi quét xong toàn bàn cờ).
3. **Rơi vật lý:** lặp "mỗi lượt, mỗi kẹo có ô trống ngay dưới thì rơi **một ô**" cho tới khi không kẹo nào di chuyển.
4. Lặp bước 1-3 cho tới khi không có ô nào bị đánh dấu.

In [8]:
def candy_crush_naive(board):
    """Cách 1: baseline chậm. Trả bàn cờ MỚI, không sửa board gốc."""
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    while True:
        marked = []
        for i in range(m):
            for j in range(n):
                v = b[i][j]
                if v == 0:
                    continue
                # dãy ngang qua (i, j)
                l = j
                while l - 1 >= 0 and b[i][l - 1] == v:
                    l -= 1
                r = j
                while r + 1 < n and b[i][r + 1] == v:
                    r += 1
                # dãy dọc qua (i, j)
                u = i
                while u - 1 >= 0 and b[u - 1][j] == v:
                    u -= 1
                d = i
                while d + 1 < m and b[d + 1][j] == v:
                    d += 1
                if r - l + 1 >= 3 or d - u + 1 >= 3:
                    marked.append((i, j))
        if not marked:
            return b
        for i, j in marked:                  # đập đồng thời
            b[i][j] = 0
        moved = True
        while moved:                         # rơi từng ô một cho tới khi yên
            moved = False
            for i in range(m - 2, -1, -1):
                for j in range(n):
                    if b[i][j] and b[i + 1][j] == 0:
                        b[i + 1][j], b[i][j] = b[i][j], 0
                        moved = True


for inp, expected in examples + extra_examples:
    got = candy_crush_naive(inp)
    print(got == expected)

True
True
True
True
True


**Độ phức tạp.** Một lượt quét: mỗi ô có thể đi tới `O(m + n)` bước ⇒ `O(mn(m + n))`. Rơi vật lý: tối đa `m` đợt, mỗi đợt `O(mn)` ⇒ `O(m²n)`. Số lượt lặp tối đa `mn / 3` (mỗi lượt đập ít nhất `3` ô). Rất chậm với bàn `50 x 50`. Space `O(mn)` cho bản sao.

**Nút thắt.** (1) Mỗi ô tự **đếm lại** dãy thay vì dùng thông tin các ô láng giềng; chỉ cần kiểm tra **mọi cửa sổ 3 ô liên tiếp**. (2) "Rơi từng ô một" lãng phí: ta biết kết quả cuối của một cột chỉ là **dồn các kẹo còn lại xuống đáy** theo đúng thứ tự, làm được trong **một lượt** với con trỏ ghi.

## 6. Ý tưởng thuật toán / bước đột phá

Bốn nhận xét:

1. **Cửa sổ 3 ô là đủ.** Mọi dãy `>= 3` kẹo liên tiếp cùng loại chứa (và bị phủ bởi) các **cửa sổ 3 ô liên tiếp** cùng loại. Vậy chỉ cần xét `board[i][j-2], board[i][j-1], board[i][j]` (ngang) và `board[i-2][j], board[i-1][j], board[i][j]` (dọc); mỗi cửa sổ đồng loại thì **đánh dấu cả 3 ô**. Dãy dài `4, 5, ...` được phủ bởi nhiều cửa sổ chồng nhau nên bị đánh dấu **toàn bộ**.
2. **Đánh dấu tại chỗ bằng số âm.** Đổi ô cần đập thành `-loại`; so sánh bằng `abs`. Nhờ vậy dãy cắt nhau vẫn bị phát hiện đầy đủ (các ô đã âm vẫn "cùng loại" khi so bằng `abs`), và ta **không cần mảng phụ**.
3. **Rơi = dồn cột bằng con trỏ ghi.** Sau khi đánh dấu xong, với mỗi cột: quét từ dưới lên, chỉ giữ ô **dương** (kẹo còn), ghi dồn xuống đáy; phần còn lại là `0`. Thứ tự các kẹo trong cột **không đổi**.
4. **Chỉ quét lại vùng bị ảnh hưởng (tối ưu).** Sau khi đập và rơi, chỉ các **cột có ô bị đập** và các **hàng từ đỉnh tới hàng thấp nhất bị đập** mới có thể xuất hiện dãy mới; phần còn lại không đổi nên không thể có dãy mới (nếu có đã bị đập từ trước). Từ đó, các lượt sau chỉ quét vùng đó.

Các nhận xét 1-3 cho Cách 2 (chuẩn); Cách 3 đổi cách biểu diễn (`groupby` + tập ô); Cách 4 thêm nhận xét 4.

## 7. Các cách tiếp theo

### Cách 2 — Đánh dấu âm tại chỗ + dồn cột bằng con trỏ ghi (cách chuẩn)

**Giải thích từng đoạn.**

- `run = True`; `while run:` mỗi vòng là một **lượt**; đầu vòng `run = False`.
- **Quét ngang:** với mỗi hàng `i`, mỗi `j` từ `2` tới `n - 1`: nếu `board[i][j]` khác `0` và `abs` của ba ô `j, j-1, j-2` bằng nhau thì `run = True` và đặt cả ba ô thành `-abs(board[i][j])`.
- **Quét dọc:** tương tự với cột `j` và `i` từ `2` tới `m - 1`.
- Nếu `run`: **rơi**: với mỗi cột `j`, `k = m - 1`; `i` từ `m - 1` lên `0`: nếu `board[i][j] > 0` thì `board[k][j] = board[i][j]`, `k -= 1`; xong điền `0` cho `k` lên `0`.
- Trả về `board`.

Vì quét ngang và dọc **không** thay đổi loại kẹo ngoài việc đổi dấu, các dãy cắt nhau đều được phát hiện đầy đủ trong cùng lượt.

In [9]:
def candy_crush_demo(board):
    """Cách 2 có in từng lượt (chạy trên bản sao)."""
    board = [row[:] for row in board]
    m, n = len(board), len(board[0])
    run, rnd = True, 0
    while run:
        run = False
        rnd += 1
        for i in range(m):
            for j in range(2, n):
                if board[i][j] and abs(board[i][j]) == abs(board[i][j - 1]) == abs(board[i][j - 2]):
                    run = True
                    board[i][j] = board[i][j - 1] = board[i][j - 2] = -abs(board[i][j])
        for j in range(n):
            for i in range(2, m):
                if board[i][j] and abs(board[i][j]) == abs(board[i - 1][j]) == abs(board[i - 2][j]):
                    run = True
                    board[i][j] = board[i - 1][j] = board[i - 2][j] = -abs(board[i][j])
        if run:
            print(f"lượt {rnd}: sau khi đánh dấu (số âm = sẽ bị đập)")
            for row in board:
                print("   ", row)
            for j in range(n):
                k = m - 1
                for i in range(m - 1, -1, -1):
                    if board[i][j] > 0:
                        board[k][j] = board[i][j]
                        k -= 1
                while k >= 0:
                    board[k][j] = 0
                    k -= 1
            print(f"lượt {rnd}: sau khi rơi")
            for row in board:
                print("   ", row)
        else:
            print(f"lượt {rnd}: không có gì để đập -> ổn định")
    return board


assert candy_crush_demo(cascade_in) == cascade_out

lượt 1: sau khi đánh dấu (số âm = sẽ bị đập)
    [2, 3, 4]
    [2, 4, 3]
    [-1, -1, -1]
    [2, 3, 4]
lượt 1: sau khi rơi
    [0, 0, 0]
    [2, 3, 4]
    [2, 4, 3]
    [2, 3, 4]
lượt 2: sau khi đánh dấu (số âm = sẽ bị đập)
    [0, 0, 0]
    [-2, 3, 4]
    [-2, 4, 3]
    [-2, 3, 4]
lượt 2: sau khi rơi
    [0, 0, 0]
    [0, 3, 4]
    [0, 4, 3]
    [0, 3, 4]
lượt 3: không có gì để đập -> ổn định


In [10]:
class Solution2:
    def candyCrush(self, board: List[List[int]]) -> List[List[int]]:
        m, n = len(board), len(board[0])
        run = True
        while run:
            run = False
            for i in range(m):
                for j in range(2, n):
                    if board[i][j] and abs(board[i][j]) == abs(board[i][j - 1]) == abs(board[i][j - 2]):
                        run = True
                        board[i][j] = board[i][j - 1] = board[i][j - 2] = -abs(board[i][j])
            for j in range(n):
                for i in range(2, m):
                    if board[i][j] and abs(board[i][j]) == abs(board[i - 1][j]) == abs(board[i - 2][j]):
                        run = True
                        board[i][j] = board[i - 1][j] = board[i - 2][j] = -abs(board[i][j])
            if run:
                for j in range(n):
                    k = m - 1
                    for i in range(m - 1, -1, -1):
                        if board[i][j] > 0:
                            board[k][j] = board[i][j]
                            k -= 1
                    while k >= 0:
                        board[k][j] = 0
                        k -= 1
        return board


for inp, expected in examples + extra_examples:
    assert Solution2().candyCrush(copy.deepcopy(inp)) == expected
print("Cách 2 đúng trên các ví dụ")

Cách 2 đúng trên các ví dụ


**Độ phức tạp Cách 2:** mỗi lượt `O(mn)` (hai lần quét + một lần rơi). Số lượt tối đa `mn / 3` (mỗi lượt đập `>= 3` ô) ⇒ Time `O(m²n²)` (rất bi quan; thực tế rất nhỏ). Space `O(1)` thêm (sửa tại chỗ).

---

### Cách 3 — `groupby` + tập ô bị đập + dựng lại từng cột

**Ý tưởng.** Cùng thuật toán nhưng **biểu diễn khác**, theo kiểu Python hơn:

- Tìm các dãy dài `>= 3` bằng `groupby` trên **từng hàng** và **từng cột** (hàm `find_runs` trả về các `(vị trí bắt đầu, độ dài)`), và gom mọi ô cần đập vào một `set` các cặp `(i, j)` (ô thuộc hai dãy cắt nhau chỉ xuất hiện một lần).
- Nếu tập rỗng thì dừng; ngược lại đặt các ô đó thành `0` rồi **dựng lại mỗi cột**: lấy các giá trị khác `0` của cột (giữ thứ tự), đệm `0` **phía trên** cho đủ độ dài `m`.

Cách này **không** đổi dấu và **không** sửa tại chỗ bàn cờ vào (làm việc trên bản sao).

In [11]:
def find_runs(values, k=3):
    """Trả danh sách (vị trí bắt đầu, độ dài) của các dãy >= k phần tử liên tiếp, bằng nhau và khác 0."""
    runs, pos = [], 0
    for value, grp in groupby(values):
        length = len(list(grp))
        if value and length >= k:
            runs.append((pos, length))
        pos += length
    return runs


print(find_runs([1, 1, 1, 2, 3, 3, 3, 3]))        # [(0, 3), (4, 4)]
print(find_runs([5, 5, 0, 0, 0, 7]))               # []   (ô trống không tính, dãy 5 chỉ dài 2)

[(0, 3), (4, 4)]
[]


In [12]:
def candy_crush_groupby(board):
    """Cách 3: groupby + tập ô bị đập + dựng lại cột. Trả bàn cờ mới."""
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    while True:
        crush = set()
        for i in range(m):
            for start, length in find_runs(b[i]):
                crush.update((i, start + t) for t in range(length))
        for j in range(n):
            column = [b[i][j] for i in range(m)]
            for start, length in find_runs(column):
                crush.update((start + t, j) for t in range(length))
        if not crush:
            return b
        for i, j in crush:
            b[i][j] = 0
        for j in range(n):
            kept = [b[i][j] for i in range(m) if b[i][j]]       # giữ thứ tự, bỏ ô trống
            kept = [0] * (m - len(kept)) + kept                 # đệm 0 phía trên
            for i in range(m):
                b[i][j] = kept[i]


for inp, expected in examples + extra_examples:
    assert candy_crush_groupby(inp) == expected
print("Cách 3 đúng trên các ví dụ")

Cách 3 đúng trên các ví dụ


**Độ phức tạp Cách 3:** mỗi lượt `O(mn)` (quét hàng, cột, dựng lại cột); số lượt tối đa `mn / 3` ⇒ Time `O(m²n²)` tệ nhất, Space `O(mn)` (bản sao + tập ô). Dễ đọc hơn Cách 2 nhưng dùng thêm bộ nhớ và hằng số lớn hơn.

---

### Cách 4 — Chỉ quét lại vùng bị ảnh hưởng sau mỗi lượt rơi

**Ý tưởng (nhận xét 4 ở mục 6).** Sau khi đập các ô và cho kẹo rơi:

- Gọi `lowest[j]` là hàng **thấp nhất** (chỉ số `i` lớn nhất) bị đập trong cột `j`. Trong cột `j`, **chỉ các ô ở hàng `0..lowest[j]`** có thể đã đổi nội dung; các ô ở dưới không đổi.
- Một dãy mới phải chứa ít nhất một ô **đã đổi**. Vậy lượt kế chỉ cần quét **các cột có ô bị đập** (nguyên cột) và **các hàng `0..R`** (với `R = max lowest[j]`, nguyên hàng). Dãy nằm hoàn toàn ở ô không đổi đã **không tồn tại** (nếu có đã bị đập ở lượt trước).
- Lượt đầu tiên quét toàn bộ bàn cờ.

Khi rơi, cũng chỉ cần dồn các hàng `0..lowest[j]` của cột `j` (phần dưới giữ nguyên).

In [13]:
def candy_crush_incremental(board):
    """Cách 4: chỉ quét lại vùng bị ảnh hưởng. Trả bàn cờ mới."""
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    rows_to_scan = range(m)
    cols_to_scan = range(n)
    while True:
        crush = set()
        for i in rows_to_scan:
            for start, length in find_runs(b[i]):
                crush.update((i, start + t) for t in range(length))
        for j in cols_to_scan:
            column = [b[i][j] for i in range(m)]
            for start, length in find_runs(column):
                crush.update((start + t, j) for t in range(length))
        if not crush:
            return b
        lowest = {}                                     # cột -> hàng thấp nhất bị đập
        for i, j in crush:
            b[i][j] = 0
            lowest[j] = max(lowest.get(j, -1), i)
        for j, low in lowest.items():                   # dồn hàng 0..low của cột j xuống
            k = low
            for i in range(low, -1, -1):
                if b[i][j]:
                    b[k][j] = b[i][j]
                    k -= 1
            while k >= 0:
                b[k][j] = 0
                k -= 1
        rows_to_scan = range(max(lowest.values()) + 1)
        cols_to_scan = sorted(lowest)


for inp, expected in examples + extra_examples:
    assert candy_crush_incremental(inp) == expected
print("Cách 4 đúng trên các ví dụ")

Cách 4 đúng trên các ví dụ


**Độ phức tạp Cách 4:** mỗi lượt chỉ quét vùng bị ảnh hưởng (nhỏ hơn `mn` khi dãy bị đập gần đáy), nên nhanh hơn nhiều trong thực tế; tệ nhất vẫn `O(mn)` mỗi lượt ⇒ `O(m²n²)`. Space `O(mn)`. Phải cẩn thận hơn: nếu quét thiếu vùng sẽ bỏ sót dãy (ta kiểm chứng ở mục 12 với hàng nghìn bàn cờ ngẫu nhiên).

## 8. TRACE từng bước

**Bảng trạng thái cho Cách 2 với `cascade_in` (`4 x 3`)** (số âm = đã đánh dấu; `.` = ô trống):

| Lượt | Giai đoạn | Bàn cờ (từng hàng) | Ghi chú |
|---|---|---|---|
| 1 | ban đầu | `[2,3,4]`, `[2,4,3]`, `[1,1,1]`, `[2,3,4]` | |
| 1 | quét ngang | hàng 2 thành `[-1,-1,-1]` | cửa sổ `j = 2`: `1,1,1` |
| 1 | quét dọc | không đổi | cột 0 là `2,2,1,2`: không có 3 liên tiếp |
| 1 | rơi | `[0,0,0]`, `[2,3,4]`, `[2,4,3]`, `[2,3,4]` | cột `j` dồn xuống |
| 2 | quét ngang | không đổi | không có hàng nào có 3 ô bằng nhau |
| 2 | quét dọc | cột 0 thành `[0,-2,-2,-2]` | cửa sổ `i = 3`: `2,2,2` |
| 2 | rơi | `[0,0,0]`, `[0,3,4]`, `[0,4,3]`, `[0,3,4]` | cột 0 toàn `0` |
| 3 | quét | không đánh dấu gì | `run = False` ⇒ dừng |

Hàm `trace_rounds` dưới đây in lại các bước đó cho **bất kỳ** bàn cờ bằng Cách 2 (đếm số lượt, số ô đập mỗi lượt).

In [14]:
def trace_rounds(board):
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    rnd = 0
    while True:
        rnd += 1
        marked = set()
        for i in range(m):
            for j in range(2, n):
                if b[i][j] and abs(b[i][j]) == abs(b[i][j - 1]) == abs(b[i][j - 2]):
                    marked.update({(i, j), (i, j - 1), (i, j - 2)})
        for j in range(n):
            for i in range(2, m):
                if b[i][j] and abs(b[i][j]) == abs(b[i - 1][j]) == abs(b[i - 2][j]):
                    marked.update({(i, j), (i - 1, j), (i - 2, j)})
        print(f"lượt {rnd}: đánh dấu {len(marked)} ô: {sorted(marked)}")
        if not marked:
            return b
        for i, j in marked:
            b[i][j] = 0
        for j in range(n):
            col = [b[i][j] for i in range(m) if b[i][j]]
            col = [0] * (m - len(col)) + col
            for i in range(m):
                b[i][j] = col[i]
        show(b, f"  sau khi đập + rơi (lượt {rnd}):")


assert trace_rounds(cascade_in) == cascade_out

lượt 1: đánh dấu 3 ô: [(2, 0), (2, 1), (2, 2)]
  sau khi đập + rơi (lượt 1):
 .  .  .
 2  3  4
 2  4  3
 2  3  4
lượt 2: đánh dấu 3 ô: [(1, 0), (2, 0), (3, 0)]
  sau khi đập + rơi (lượt 2):
 .  .  .
 .  3  4
 .  4  3
 .  3  4
lượt 3: đánh dấu 0 ô: []


**Trace cho dấu cộng `cross_in`:** lượt 1 đánh dấu **5** ô (ngang: `(1,0),(1,1),(1,2)`; dọc: `(0,0),(1,0),(2,0)`; ô `(1,0)` chỉ tính một lần). Đập đồng thời rồi rơi: cột 0 trống hoàn toàn; cột 1 `[2,.,3]` thành `[.,2,3]`; cột 2 `[3,.,2]` thành `[.,3,2]`. Lượt 2: không còn dãy. Kết quả `[[0,0,0],[0,2,3],[0,3,2]]`.

In [15]:
res = trace_rounds(cross_in)
assert res == cross_out

lượt 1: đánh dấu 5 ô: [(0, 0), (1, 0), (1, 1), (1, 2), (2, 0)]
  sau khi đập + rơi (lượt 1):
 .  .  .
 .  2  3
 .  3  2
lượt 2: đánh dấu 0 ô: []


## 9. Code nộp LeetCode

Ta nộp **Cách 2**: đánh dấu âm tại chỗ + dồn cột bằng con trỏ ghi; `O(1)` bộ nhớ thêm, ngắn, và là lời giải chuẩn. Nó **sửa trực tiếp** `board` và trả về chính nó (LeetCode chấp nhận).

In [16]:
class Solution:
    def candyCrush(self, board: List[List[int]]) -> List[List[int]]:
        m, n = len(board), len(board[0])
        run = True
        while run:
            run = False
            for i in range(m):                        # 1) đánh dấu dãy ngang (cửa sổ 3 ô)
                for j in range(2, n):
                    if board[i][j] and abs(board[i][j]) == abs(board[i][j - 1]) == abs(board[i][j - 2]):
                        run = True
                        board[i][j] = board[i][j - 1] = board[i][j - 2] = -abs(board[i][j])
            for j in range(n):                        # 2) đánh dấu dãy dọc
                for i in range(2, m):
                    if board[i][j] and abs(board[i][j]) == abs(board[i - 1][j]) == abs(board[i - 2][j]):
                        run = True
                        board[i][j] = board[i - 1][j] = board[i - 2][j] = -abs(board[i][j])
            if run:                                   # 3) đập (số âm) + rơi: dồn ô dương xuống đáy
                for j in range(n):
                    k = m - 1
                    for i in range(m - 1, -1, -1):
                        if board[i][j] > 0:
                            board[k][j] = board[i][j]
                            k -= 1
                    while k >= 0:
                        board[k][j] = 0
                        k -= 1
        return board


for inp, expected in examples + extra_examples:
    assert Solution().candyCrush(copy.deepcopy(inp)) == expected
print("Code nộp đúng trên 5 ví dụ")

Code nộp đúng trên 5 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Một lượt "đánh dấu" tìm đúng mọi ô cần đập.** Một ô `(i, j)` cần đập khi nó nằm trong một dãy `>= 3` kẹo cùng loại **liền nhau** theo hàng hoặc cột. Mọi dãy như vậy chứa ít nhất một cửa sổ 3 ô liên tiếp cùng loại, và mọi ô của dãy thuộc **ít nhất một** cửa sổ như thế (ô đầu thuộc cửa sổ đầu, ô cuối thuộc cửa sổ cuối, ô giữa thuộc cửa sổ chứa nó). Vì ta quét **mọi** cửa sổ và đánh dấu cả ba ô, mọi ô cần đập đều bị đánh dấu; ngược lại ô bị đánh dấu thuộc một cửa sổ cùng loại nên đúng cần đập: **không dư, không thiếu**.

**(b) Dấu âm không làm sai phép so sánh.** Ô đã đánh dấu mang `-v` và ta so `abs`, nên một dãy ngang và một dãy dọc cắt nhau đều được nhận ra trong **cùng một lượt**, đúng yêu cầu "đập tất cả cùng một lúc". Ô trống (`0`) bị loại bằng điều kiện `board[i][j]` khác `0` (nếu không, ba ô trống liên tiếp bị coi là "dãy").

**(c) Rơi đúng.** Kẹo trong một cột chỉ di chuyển **dọc xuống** và **không vượt qua nhau**; vì vậy kết quả sau khi rơi là: các kẹo **còn lại** (dương) của cột theo đúng **thứ tự trên xuống** ban đầu, dồn sát đáy; phần trên là `0`. Con trỏ ghi `k` chạy từ đáy lên, ghi mỗi kẹo dương vào vị trí `k` (luôn `k >= i` nên không ghi đè kẹo chưa đọc), rồi điền `0`. Đây chính là phép "rơi" của luật.

**(d) Kết thúc.** Mỗi lượt có đập đều **loại ít nhất 3 kẹo** khỏi bàn cờ (không có kẹo mới), nên số lượt tối đa `mn / 3`; lượt không đánh dấu gì thì bàn cờ **ổn định** (không còn dãy `>= 3` và, vì vừa rơi, không còn kẹo lơ lửng). Đáp án là duy nhất vì mọi bước đều xác định.

**(e) Cách 3 tương đương** vì `find_runs` tìm đúng các dãy cực đại `>= 3` (bằng `groupby`), tập `set` hợp nhất theo ô, và "dựng lại cột" là cùng phép rơi. **Cách 4** đúng vì (nhận xét 4): các ô dưới hàng thấp nhất bị đập của mỗi cột không đổi nội dung; một dãy mới phải chứa ô **đã đổi** (nếu không nó đã tồn tại và bị đập ở lượt trước), mà các ô đã đổi chỉ nằm ở các cột bị đập, hàng `0..R`. Ta quét mọi hàng `0..R` đầy đủ và mọi cột bị đập đầy đủ nên **bắt trọn**. Cách 1 là mô phỏng thô dùng làm oracle; ở mục 12 cả bốn cách được đối chiếu trên hàng nghìn bàn cờ ngẫu nhiên và kiểm tra các **tính chất bắt buộc** của đáp án.

## 11. Phân tích Time / Space Complexity

Gọi `m x n` là kích thước, `R` là số lượt (`R <= mn / 3`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng vật lý chậm | `O(R · (mn(m+n) + m²n))` | `O(mn)` | chỉ làm oracle |
| 2. Đánh dấu âm + dồn cột (nộp) | `O(R · mn)` ⊂ `O(m²n²)` | `O(1)` thêm | sửa tại chỗ |
| 3. `groupby` + tập ô | `O(R · mn)` | `O(mn)` | dễ đọc, nhiều bộ nhớ hơn |
| 4. Quét lại vùng bị ảnh hưởng | `O(R · mn)` tệ nhất, thường nhỏ hơn | `O(mn)` | khó viết đúng nhất |

Với `m, n <= 50` (`mn <= 2500`), một lượt cỡ vài nghìn phép so sánh; số lượt thực tế rất nhỏ nên mọi cách trừ Cách 1 đều nhanh.

## 12. Bộ test

### 12.1. Ví dụ đề, ca tính tay và biên

| Bàn cờ | Mô tả | Kết quả (tính tay) |
|---|---|---|
| ví dụ 1, ví dụ 2 | từ đề | như đề |
| `cascade_in` | đập hàng, cột rơi tạo dãy mới | `[[0,0,0],[0,3,4],[0,4,3],[0,3,4]]` |
| `cross_in` | hai dãy cắt nhau (dấu cộng) | `[[0,0,0],[0,2,3],[0,3,2]]` |
| `[[1,1,1],[2,3,2],[3,2,3]]` | một dãy ngang ở hàng trên cùng | `[[0,0,0],[2,3,2],[3,2,3]]` |
| `[[1,2,3],[1,3,2],[1,2,3]]` | một dãy dọc ở cột 0 | `[[0,2,3],[0,3,2],[0,2,3]]` |
| `[[2,3,3,4],[2,5,6,4],[2,7,8,4]]` | hai dãy dọc song song | `[[0,3,3,0],[0,5,6,0],[0,7,8,0]]` |
| `[[1,1,1,1,1],[2,3,2,3,2],[3,2,3,2,3]]` | dãy dài `5` | hàng đầu thành `0` |
| `[[7,7,7],[7,7,7],[7,7,7]]` | toàn một loại | toàn `0` |
| `[[1,2,1],[2,1,2],[1,2,1]]` | đã ổn định | giữ nguyên |

In [17]:
hand_cases = [
    ([[1, 1, 1], [2, 3, 2], [3, 2, 3]], [[0, 0, 0], [2, 3, 2], [3, 2, 3]]),
    ([[1, 2, 3], [1, 3, 2], [1, 2, 3]], [[0, 2, 3], [0, 3, 2], [0, 2, 3]]),
    ([[2, 3, 3, 4], [2, 5, 6, 4], [2, 7, 8, 4]], [[0, 3, 3, 0], [0, 5, 6, 0], [0, 7, 8, 0]]),
    ([[1, 1, 1, 1, 1], [2, 3, 2, 3, 2], [3, 2, 3, 2, 3]], [[0, 0, 0, 0, 0], [2, 3, 2, 3, 2], [3, 2, 3, 2, 3]]),
    ([[7, 7, 7], [7, 7, 7], [7, 7, 7]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]]),
    ([[1, 2, 1], [2, 1, 2], [1, 2, 1]], [[1, 2, 1], [2, 1, 2], [1, 2, 1]]),
]
all_cases = examples + extra_examples + hand_cases


def run_solution2(b):
    return Solution2().candyCrush(copy.deepcopy(b))


def run_submitted(b):
    return Solution().candyCrush(copy.deepcopy(b))


solvers = {
    "Cách 1 vật lý chậm": candy_crush_naive,
    "Cách 2 âm tại chỗ": run_solution2,
    "Cách 3 groupby": candy_crush_groupby,
    "Cách 4 quét vùng": candy_crush_incremental,
    "Code nộp": run_submitted,
}
for inp, expected in all_cases:
    snapshot = copy.deepcopy(inp)
    for name, f in solvers.items():
        assert f(inp) == expected, (name, inp)
    assert inp == snapshot, "các cách (trừ code nộp trên bản sao) không được làm hỏng input"
print("Tất cả", len(all_cases), "ca đều đúng với mọi cách")

Tất cả 11 ca đều đúng với mọi cách


### 12.2. Các tính chất bắt buộc của một đáp án đúng

Với **mọi** bàn cờ, đáp án phải thỏa các tính chất sau (đây là kiểm tra **độc lập** với thuật toán):

1. **Cùng kích thước.**
2. **Ổn định về đập:** không còn dãy `>= 3` kẹo cùng loại (khác `0`) theo hàng hay cột.
3. **Ổn định về rơi:** trong mỗi cột, không có ô `0` nằm **dưới** một kẹo (mọi `0` đều ở phía trên).
4. **Bảo toàn:** các kẹo còn lại của mỗi cột là một **dãy con** (giữ thứ tự) của cột ban đầu (kẹo chỉ bị đập hoặc rơi, không đổi thứ tự trong cột, không có kẹo mới).
5. **Không thể "đập thêm" nếu đã ổn định:** chạy lại thuật toán trên kết quả không đổi gì.

In [18]:
def has_run(values, k=3):
    return any(v and sum(1 for _ in g) >= k for v, g in groupby(values))


def is_subsequence(small, big):
    it = iter(big)
    return all(x in it for x in small)


def check_properties(original, result):
    m, n = len(original), len(original[0])
    assert len(result) == m and all(len(r) == n for r in result)                  # 1
    for i in range(m):
        assert not has_run(result[i]), ("còn dãy ngang", i)                       # 2
    for j in range(n):
        col = [result[i][j] for i in range(m)]
        assert not has_run(col), ("còn dãy dọc", j)                               # 2
        seen_candy = False
        for v in col:                                                             # 3: sau kẹo đầu tiên không còn 0
            if v:
                seen_candy = True
            else:
                assert not seen_candy, ("kẹo lơ lửng", j)
        orig_col = [v for v in (original[i][j] for i in range(m)) if v]
        res_col = [v for v in col if v]
        assert is_subsequence(res_col, orig_col), ("sai thứ tự/kẹo lạ", j)        # 4
    assert run_solution2(result) == result                                        # 5


for inp, expected in all_cases:
    check_properties(inp, expected)
print("Các tính chất đúng trên mọi đáp án mong đợi đã biết")

Các tính chất đúng trên mọi đáp án mong đợi đã biết


### 12.3. Đối chiếu ngẫu nhiên có seed (bàn cờ nhỏ, so cả bốn cách và Cách 1)

Bàn cờ ngẫu nhiên `3..8` hàng/cột, số loại kẹo `2..4` (ít loại để có nhiều dãy và nhiều phản ứng dây chuyền), đúng ràng buộc (`>= 1`). Cách 1 (vật lý chậm) là oracle.

In [19]:
def random_board(rng, m, n, k):
    return [[rng.randint(1, k) for _ in range(n)] for _ in range(m)]


rng = random.Random(723)
for _ in range(3000):
    m, n = rng.randint(3, 8), rng.randint(3, 8)
    k = rng.randint(2, 4)
    b = random_board(rng, m, n, k)
    ref = candy_crush_naive(b)
    for name, f in solvers.items():
        assert f(b) == ref, (name, b)
    check_properties(b, ref)
print("Đã đối chiếu 3000 bàn cờ ngẫu nhiên (seed 723): 5 cách khớp và thỏa các tính chất")

Đã đối chiếu 3000 bàn cờ ngẫu nhiên (seed 723): 5 cách khớp và thỏa các tính chất


Để chắc là bộ test thật sự có **phản ứng dây chuyền** (không chỉ một lượt), ta đếm số lượt và số ô đập trên các bàn cờ ngẫu nhiên:

In [20]:
def count_rounds(board):
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    rounds = 0
    while True:
        marked = set()
        for i in range(m):
            for start, length in find_runs(b[i]):
                marked.update((i, start + t) for t in range(length))
        for j in range(n):
            for start, length in find_runs([b[i][j] for i in range(m)]):
                marked.update((start + t, j) for t in range(length))
        if not marked:
            return rounds
        rounds += 1
        for i, j in marked:
            b[i][j] = 0
        for j in range(n):
            col = [b[i][j] for i in range(m) if b[i][j]]
            col = [0] * (m - len(col)) + col
            for i in range(m):
                b[i][j] = col[i]


rng = random.Random(7230)
hist = Counter()
for _ in range(2000):
    b = random_board(rng, rng.randint(3, 8), rng.randint(3, 8), rng.randint(2, 4))
    hist[count_rounds(b)] += 1
print(sorted(hist.items()))
assert max(hist) >= 4 and sum(c for r, c in hist.items() if r >= 2) > 300

[(0, 185), (1, 1112), (2, 573), (3, 100), (4, 27), (5, 3)]


### 12.4. Cỡ lớn (đúng ràng buộc `50 x 50`)

Bàn cờ lớn với ít loại kẹo (nhiều phản ứng dây chuyền), nhiều loại kẹo (ít đập), và các dạng cực đoan. Cách 1 chạy chậm nên bỏ qua; so Cách 2, 3, 4 và kiểm tra các tính chất.

In [21]:
rng = random.Random(72300)
boards = []
for k in (2, 3, 5, 20, 2000):
    boards.append(random_board(rng, 50, 50, k))
boards.append([[1] * 50 for _ in range(50)])                                   # toàn một loại
boards.append([[(i + j) % 2 + 1 for j in range(50)] for i in range(50)])      # bàn cờ caro: ổn định sẵn
boards.append([[1 + (i // 3 + j) % 7 for j in range(50)] for i in range(50)])  # dãy dọc ngắn xen kẽ
for b in boards:
    ref = Solution2().candyCrush(copy.deepcopy(b))
    assert candy_crush_groupby(b) == ref == candy_crush_incremental(b) == run_submitted(b)
    check_properties(b, ref)
    print("rounds =", count_rounds(b), "| ô trống sau cùng:", sum(v == 0 for row in ref for v in row))

rounds = 7 | ô trống sau cùng: 2376
rounds = 11 | ô trống sau cùng: 1988


rounds = 10 | ô trống sau cùng: 1152
rounds = 2 | ô trống sau cùng: 40
rounds = 0 | ô trống sau cùng: 0
rounds = 1 | ô trống sau cùng: 2500
rounds = 0 | ô trống sau cùng: 0
rounds = 1 | ô trống sau cùng: 2400


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đập **ngay khi** thấy dãy (không đánh dấu hết rồi mới đập)

Nếu đập hàng trước (đặt `0` ngay) rồi mới quét cột, dãy dọc cắt hàng đó bị phá và **không** được đập, trái với luật "đập tất cả cùng một lúc".

In [22]:
def sai_dap_ngay(board):
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    while True:
        changed = False
        for i in range(m):
            for j in range(2, n):
                if b[i][j] and b[i][j] == b[i][j - 1] == b[i][j - 2]:
                    b[i][j] = b[i][j - 1] = b[i][j - 2] = 0          # SAI: đập ngay
                    changed = True
        for j in range(n):
            for i in range(2, m):
                if b[i][j] and b[i][j] == b[i - 1][j] == b[i - 2][j]:
                    b[i][j] = b[i - 1][j] = b[i - 2][j] = 0
                    changed = True
        if not changed:
            return b
        for j in range(n):
            col = [b[i][j] for i in range(m) if b[i][j]]
            col = [0] * (m - len(col)) + col
            for i in range(m):
                b[i][j] = col[i]


print("sai :", sai_dap_ngay(cross_in))
print("đúng:", Solution().candyCrush(copy.deepcopy(cross_in)))
assert sai_dap_ngay(cross_in) != cross_out

sai : [[0, 0, 0], [1, 2, 3], [1, 3, 2]]
đúng: [[0, 0, 0], [0, 2, 3], [0, 3, 2]]


**Vì sao sai:** ở dấu cộng, hàng 1 bị xóa trước thì cột 0 còn `1, 0, 1` và không còn là dãy `3`; hai ô `(0,0)` và `(2,0)` sống sót sai. **Sửa:** quét **toàn bộ** trước, đánh dấu (số âm hoặc `set`), rồi mới đập và cho rơi.

### Lỗi 2: Đánh dấu âm nhưng quên `abs` khi so sánh

Nếu so `board[i][j] == board[i][j-1] == board[i][j-2]` mà không `abs`, ô đã bị đánh dấu (âm) sẽ không còn bằng ô dương cùng loại, nên dãy cắt nhau bị bỏ sót.

In [23]:
def sai_khong_abs(board):
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    run = True
    while run:
        run = False
        for i in range(m):
            for j in range(2, n):
                if b[i][j] and b[i][j] == b[i][j - 1] == b[i][j - 2]:      # SAI: thiếu abs
                    run = True
                    b[i][j] = b[i][j - 1] = b[i][j - 2] = -abs(b[i][j])
        for j in range(n):
            for i in range(2, m):
                if b[i][j] and b[i][j] == b[i - 1][j] == b[i - 2][j]:
                    run = True
                    b[i][j] = b[i - 1][j] = b[i - 2][j] = -abs(b[i][j])
        if run:
            for j in range(n):
                k = m - 1
                for i in range(m - 1, -1, -1):
                    if b[i][j] > 0:
                        b[k][j] = b[i][j]
                        k -= 1
                while k >= 0:
                    b[k][j] = 0
                    k -= 1
    return b


print("sai :", sai_khong_abs(cross_in))
print("đúng:", Solution().candyCrush(copy.deepcopy(cross_in)))
assert sai_khong_abs(cross_in) != cross_out

sai : [[0, 0, 0], [1, 2, 3], [1, 3, 2]]
đúng: [[0, 0, 0], [0, 2, 3], [0, 3, 2]]


**Vì sao sai:** sau khi đánh dấu hàng 1 thành `-1 -1 -1`, cột 0 là `1, -1, 1` không bằng nhau nên dãy dọc bị bỏ sót (còn `(0,0)` và `(2,0)`). **Sửa:** so sánh bằng `abs(...)`.

### Lỗi 3: Chỉ rơi một lượt (quên **lặp** tới khi ổn định)

Sau khi rơi có thể xuất hiện dãy mới (cascade). Dừng sau một lượt sẽ sai.

In [24]:
def sai_mot_luot(board):
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    marked = set()
    for i in range(m):
        for start, length in find_runs(b[i]):
            marked.update((i, start + t) for t in range(length))
    for j in range(n):
        for start, length in find_runs([b[i][j] for i in range(m)]):
            marked.update((start + t, j) for t in range(length))
    for i, j in marked:
        b[i][j] = 0
    for j in range(n):
        col = [b[i][j] for i in range(m) if b[i][j]]
        col = [0] * (m - len(col)) + col
        for i in range(m):
            b[i][j] = col[i]
    return b                                   # SAI: không lặp


print("sai :", sai_mot_luot(cascade_in))
print("đúng:", Solution().candyCrush(copy.deepcopy(cascade_in)))
assert sai_mot_luot(cascade_in) != cascade_out

sai : [[0, 0, 0], [2, 3, 4], [2, 4, 3], [2, 3, 4]]
đúng: [[0, 0, 0], [0, 3, 4], [0, 4, 3], [0, 3, 4]]


**Vì sao sai:** sau lượt 1, cột 0 là `0, 2, 2, 2` (dãy mới) nhưng ta dừng. **Sửa:** `while run:` lặp tới khi một lượt không đánh dấu gì.

### Lỗi 4: Rơi sai bằng một lượt hoán đổi

Viết "nếu ô dưới trống thì đổi chỗ" chỉ **một lượt từ trên xuống** chỉ cho mỗi kẹo rơi tối đa một ô (hoặc dây chuyền bị rối), để lại kẹo lơ lửng.

In [25]:
def sai_roi_mot_luot_hoan_doi(col):
    col = col[:]
    for i in range(len(col) - 1):                 # quét trên xuống, mỗi kẹo chỉ đổi chỗ 1 lần
        if col[i] and col[i + 1] == 0:
            col[i], col[i + 1] = 0, col[i]
    return col


def roi_dung(col):
    kept = [v for v in col if v > 0]
    return [0] * (len(col) - len(kept)) + kept


c = [1, 2, 0, 0]
print("sai :", sai_roi_mot_luot_hoan_doi(c))       # [1, 0, 0, 2]: kẹo 1 vẫn lơ lửng
print("đúng:", roi_dung(c))                          # [0, 0, 1, 2]
assert sai_roi_mot_luot_hoan_doi(c) != roi_dung(c)

sai : [1, 0, 0, 2]
đúng: [0, 0, 1, 2]


**Vì sao sai:** kẹo `2` rơi xuống đáy nhưng kẹo `1` ở trên đã bị xét **trước** khi có chỗ trống nên đứng yên. **Sửa:** quét **từ dưới lên** với con trỏ ghi (dồn tất cả kẹo xuống đáy), hoặc xây list "các ô khác 0" rồi đệm `0` phía trên.

### Lỗi 5: Quên một trong hai hướng (chỉ quét hàng hoặc chỉ quét cột)

Quên quét cột (hoặc hàng) làm bỏ sót các dãy theo hướng còn lại.

In [26]:
def sai_chi_quet_hang(board):
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    while True:
        marked = set()
        for i in range(m):
            for start, length in find_runs(b[i]):
                marked.update((i, start + t) for t in range(length))
        if not marked:                              # SAI: thiếu quét cột
            return b
        for i, j in marked:
            b[i][j] = 0
        for j in range(n):
            col = [b[i][j] for i in range(m) if b[i][j]]
            col = [0] * (m - len(col)) + col
            for i in range(m):
                b[i][j] = col[i]


col_case = [[1, 2, 3], [1, 3, 2], [1, 2, 3]]
print("sai :", sai_chi_quet_hang(col_case))
print("đúng:", Solution().candyCrush(copy.deepcopy(col_case)))
assert sai_chi_quet_hang(col_case) != [[0, 2, 3], [0, 3, 2], [0, 2, 3]]

sai : [[1, 2, 3], [1, 3, 2], [1, 2, 3]]
đúng: [[0, 2, 3], [0, 3, 2], [0, 2, 3]]


**Sửa:** quét **cả hai hướng** trong cùng lượt; ô thuộc cả hai hướng chỉ cần được đánh dấu (dùng số âm hoặc `set`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đập đồng thời", "rơi xuống", "lặp tới khi ổn định", lưới `m x n` ⇒ **mô phỏng theo lượt**: *đánh dấu → áp dụng → lặp*.
- **Tách hai pha trong mỗi lượt:** pha tìm (không sửa gì ngoài đánh dấu) và pha áp dụng. Điều này đảm bảo tính **đồng thời**.
- **Đánh dấu bằng số âm + `abs`** là thủ thuật tiết kiệm bộ nhớ rất hay gặp (Game of Life cũng dùng mã trạng thái tạm).
- **Trọng lực = "dồn" bằng hai con trỏ**: con trỏ ghi chạy từ đáy lên, con trỏ đọc cũng từ đáy lên; đây là mẫu của *Move Zeroes*, *Remove Element*.
- **Cửa sổ 3 ô** đủ để bắt mọi dãy `>= 3`: không cần đếm độ dài dãy.
- **Số lượt hữu hạn:** mỗi lượt giảm số kẹo ít nhất `3`, nên vòng lặp luôn dừng. Hãy nói rõ điều này khi phân tích.
- **Kiểm chứng bằng tính chất bắt buộc** (ổn định, bảo toàn thứ tự, không lơ lửng) rất hữu ích khi không có oracle thứ hai.
- Bài liên quan: **Game of Life** (cập nhật đồng thời), **Move Zeroes**, **Candy Crush 1D** (bài một chiều, dùng ngăn xếp), **Flood Fill**.

## 15. Biến thể

1. **Đếm tổng số kẹo bị đập** (điểm số) và **số lượt** (độ sâu phản ứng dây chuyền).
2. **Ngưỡng `k` khác 3** (đập khi có `k` kẹo liền nhau): thay cửa sổ `3` bằng `k` (hoặc dùng `find_runs(values, k)`).
3. **Chỉ trả về bàn cờ sau đúng một lượt** (đập + rơi một lần).
4. **Candy Crush 1D** (chuỗi): xóa các đoạn liên tiếp `>= 3` rồi nối lại, lặp; dùng ngăn xếp `(ký tự, số lượng)`.
5. **Tìm nước đi:** hoán đổi hai ô kề nhau sao cho tạo ra dãy `>= 3` (bài thiết kế trò chơi thật).
6. **Có kẹo mới rơi từ mép trên** (ngẫu nhiên): bài mô phỏng đầy đủ của trò chơi.

In [27]:
def candy_crush_stats(board, k=3):
    """Biến thể 1, 2: trả (bàn cờ ổn định, số lượt, tổng số kẹo bị đập) với ngưỡng k."""
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    rounds = total = 0
    while True:
        marked = set()
        for i in range(m):
            for start, length in find_runs(b[i], k):
                marked.update((i, start + t) for t in range(length))
        for j in range(n):
            for start, length in find_runs([b[i][j] for i in range(m)], k):
                marked.update((start + t, j) for t in range(length))
        if not marked:
            return b, rounds, total
        rounds += 1
        total += len(marked)
        for i, j in marked:
            b[i][j] = 0
        for j in range(n):
            col = [b[i][j] for i in range(m) if b[i][j]]
            col = [0] * (m - len(col)) + col
            for i in range(m):
                b[i][j] = col[i]


final, rounds, total = candy_crush_stats(cascade_in)
print(rounds, total)                                    # 2 6  (3 ô hàng + 3 ô cột)
assert final == cascade_out and (rounds, total) == (2, 6)
final4, r4, t4 = candy_crush_stats(cross_in)
print(r4, t4)                                           # 1 5
assert (r4, t4) == (1, 5) and final4 == cross_out
fin_k4, _, _ = candy_crush_stats([[1, 1, 1, 2], [2, 2, 3, 3], [3, 1, 1, 1]], k=4)
print(fin_k4)                                           # ngưỡng 4: không có gì bị đập

2 6
1 5
[[1, 1, 1, 2], [2, 2, 3, 3], [3, 1, 1, 1]]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng vật lý chậm | Mỗi ô đếm dãy quanh nó; kẹo rơi từng ô một | `O(R · (mn(m+n) + m²n))` | `O(mn)` | Làm oracle, hiểu luật |
| 2. Đánh dấu âm + dồn cột | Cửa sổ 3 ô, đổi dấu, con trỏ ghi dồn xuống | `O(R · mn)` | `O(1)` thêm | **Nộp bài**, tiết kiệm bộ nhớ |
| 3. `groupby` + tập ô + dựng cột | Tìm dãy cực đại bằng `groupby`, gom ô bằng `set`, dựng lại cột | `O(R · mn)` | `O(mn)` | Dễ đọc, dễ mở rộng ngưỡng `k` |
| 4. Quét lại vùng ảnh hưởng | Chỉ quét cột bị đập và hàng từ đỉnh tới hàng thấp nhất bị đập | `O(R · mn)` tệ nhất, thường nhỏ hơn | `O(mn)` | Bàn cờ lớn, nhiều lượt; khó viết đúng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `find_runs([1, 1, 1, 2, 2, 2, 2])`; `find_runs([5, 5, 0, 0, 0])`; kết quả dồn cột `[3, 0, 4, 0]` về đáy; `abs(-7) == abs(7)`; `[[0] * 2 for _ in range(2)]`.

**Bài 2 (tính tay).** Với bàn cờ `3 x 4`: `[[2,3,3,4],[2,5,6,4],[2,7,8,4]]`, tìm các ô bị đập ở lượt 1 và bàn cờ cuối cùng. Vì sao dừng ở lượt 2?

**Bài 3 (điền TODO).** Hoàn thành `gravity_todo(board)`: cho kẹo rơi (chỉ giữ ô dương, dồn xuống đáy mỗi cột, còn lại `0`).

In [28]:
def gravity_todo(board):
    # TODO 1: m, n = len(board), len(board[0])
    # TODO 2: với mỗi cột j: k = m - 1; for i từ m - 1 xuống 0: nếu board[i][j] > 0 thì board[k][j] = board[i][j]; k -= 1
    # TODO 3: while k >= 0: board[k][j] = 0; k -= 1
    # TODO 4: return board
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `candy_crush_rewrite(board)` theo Cách 2 (đánh dấu âm bằng `abs`, hai lần quét, dồn cột), trả về bàn cờ đã sửa tại chỗ.

**Bài 5 (biến thể / mini challenge).** (a) Viết `crush_one_round(board)` trả `(bàn cờ sau đúng một lượt, có_đập_không)`; kiểm chứng: lặp `crush_one_round` tới khi `có_đập_không == False` phải cho đúng kết quả đầy đủ. (b) Viết `count_cells_by_round(board)` trả danh sách **số ô bị đập ở mỗi lượt** (độ dài bằng số lượt); kiểm chứng bằng `candy_crush_stats`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[(0, 3), (3, 4)]`; `[]` (ô `0` không tính, dãy `5` chỉ dài `2`); `[0, 0, 3, 4]`; `True`; `[[0, 0], [0, 0]]`. Bài 3: bám sát mục 3.3.
- **Mức 2 (vừa):** Bài 2: cột 0 là `2,2,2` và cột 3 là `4,4,4` ⇒ đập `6` ô: `(0,0),(1,0),(2,0),(0,3),(1,3),(2,3)`; sau đó cột 0 và cột 3 toàn `0`, các cột 1, 2 không đổi (không có ô nào ở trên cần rơi). Lượt 2: hàng `[0,3,3,0]` chỉ có hai số `3` liền nhau nên không phải dãy `3`; không có gì để đập, dừng. Bài 4: nhớ `-abs(...)` khi đánh dấu và `board[i][j] > 0` khi dồn.
- **Mức 3 (sâu):** Bài 5(a): chép Cách 3 nhưng chỉ làm **một** vòng `while` rồi trả `(b, bool(crush))`. Bài 5(b): trong vòng `while` của Cách 3, thêm `sizes.append(len(crush))` ngay trước khi đập. So sánh `len(sizes)` với số lượt, `sum(sizes)` với `total` từ `candy_crush_stats`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert find_runs([1, 1, 1, 2, 2, 2, 2]) == [(0, 3), (3, 4)]
assert find_runs([5, 5, 0, 0, 0]) == []
assert roi_dung([3, 0, 4, 0]) == [0, 0, 3, 4]
assert abs(-7) == abs(7)
assert [[0] * 2 for _ in range(2)] == [[0, 0], [0, 0]]

# Bài 2
b2 = [[2, 3, 3, 4], [2, 5, 6, 4], [2, 7, 8, 4]]
assert Solution().candyCrush(copy.deepcopy(b2)) == [[0, 3, 3, 0], [0, 5, 6, 0], [0, 7, 8, 0]]
fin2, r2, t2 = candy_crush_stats(b2)
assert (r2, t2) == (1, 6)


# Bài 3
def gravity_todo_ans(board):
    m, n = len(board), len(board[0])
    for j in range(n):
        k = m - 1
        for i in range(m - 1, -1, -1):
            if board[i][j] > 0:
                board[k][j] = board[i][j]
                k -= 1
        while k >= 0:
            board[k][j] = 0
            k -= 1
    return board


assert gravity_todo_ans([[1, -2], [0, 3], [4, 0]]) == [[0, 0], [1, 0], [4, 3]]


# Bài 4
def candy_crush_rewrite(board):
    m, n = len(board), len(board[0])
    run = True
    while run:
        run = False
        for i in range(m):
            for j in range(2, n):
                if board[i][j] and abs(board[i][j]) == abs(board[i][j - 1]) == abs(board[i][j - 2]):
                    run = True
                    board[i][j] = board[i][j - 1] = board[i][j - 2] = -abs(board[i][j])
        for j in range(n):
            for i in range(2, m):
                if board[i][j] and abs(board[i][j]) == abs(board[i - 1][j]) == abs(board[i - 2][j]):
                    run = True
                    board[i][j] = board[i - 1][j] = board[i - 2][j] = -abs(board[i][j])
        if run:
            gravity_todo_ans(board)
    return board


# Bài 5
def crush_one_round(board):
    b = [row[:] for row in board]
    m, n = len(b), len(b[0])
    crush = set()
    for i in range(m):
        for start, length in find_runs(b[i]):
            crush.update((i, start + t) for t in range(length))
    for j in range(n):
        for start, length in find_runs([b[i][j] for i in range(m)]):
            crush.update((start + t, j) for t in range(length))
    for i, j in crush:
        b[i][j] = 0
    gravity_todo_ans(b)
    return b, bool(crush)


def count_cells_by_round(board):
    sizes, cur = [], board
    while True:
        m, n = len(cur), len(cur[0])
        crush = set()
        for i in range(m):
            for start, length in find_runs(cur[i]):
                crush.update((i, start + t) for t in range(length))
        for j in range(n):
            for start, length in find_runs([cur[i][j] for i in range(m)]):
                crush.update((start + t, j) for t in range(length))
        if not crush:
            return sizes
        sizes.append(len(crush))
        cur, _ = crush_one_round(cur)


rng5 = random.Random(7231)
for _ in range(400):
    b = random_board(rng5, rng5.randint(3, 8), rng5.randint(3, 8), rng5.randint(2, 4))
    ref = candy_crush_naive(b)
    assert candy_crush_rewrite(copy.deepcopy(b)) == ref
    cur, changed = b, True
    while changed:
        cur, changed = crush_one_round(cur)
    assert cur == ref
    sizes = count_cells_by_round(b)
    _, rounds, total = candy_crush_stats(b)
    assert len(sizes) == rounds and sum(sizes) == total
assert count_cells_by_round(cascade_in) == [3, 3]
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đánh dấu âm → đập + rơi → lặp)

```python
class Solution:
    def candyCrush(self, board: List[List[int]]) -> List[List[int]]:
        m, n = len(board), len(board[0])
        run = True
        while run:
            run = False
            for i in range(m):                                   # dãy ngang: cửa sổ 3 ô
                for j in range(2, n):
                    if board[i][j] and abs(board[i][j]) == abs(board[i][j - 1]) == abs(board[i][j - 2]):
                        run = True
                        board[i][j] = board[i][j - 1] = board[i][j - 2] = -abs(board[i][j])
            for j in range(n):                                   # dãy dọc
                for i in range(2, m):
                    if board[i][j] and abs(board[i][j]) == abs(board[i - 1][j]) == abs(board[i - 2][j]):
                        run = True
                        board[i][j] = board[i - 1][j] = board[i - 2][j] = -abs(board[i][j])
            if run:                                              # rơi: dồn ô dương xuống đáy
                for j in range(n):
                    k = m - 1
                    for i in range(m - 1, -1, -1):
                        if board[i][j] > 0:
                            board[k][j] = board[i][j]
                            k -= 1
                    while k >= 0:
                        board[k][j] = 0
                        k -= 1
        return board
```

**Checklist**

- [ ] **Đánh dấu hết rồi mới đập** (không đập ngay khi thấy), để các dãy cắt nhau cùng bị đập?
- [ ] So sánh bằng `abs(...)` và loại ô trống bằng `board[i][j]` khác `0`?
- [ ] Quét **cả hai hướng** (hàng và cột) trong cùng một lượt?
- [ ] Rơi bằng con trỏ ghi **từ dưới lên**, chỉ giữ ô `> 0`, phần còn lại điền `0`?
- [ ] **Lặp** tới khi một lượt không đánh dấu gì (có thể có phản ứng dây chuyền)?
- [ ] Không có kẹo mới rơi vào từ mép trên?
- [ ] Đã thử ví dụ 1, ví dụ 2, dấu cộng (hàng cắt cột), cascade và bàn cờ đã ổn định?
- [ ] Skeleton đúng `class Solution` / `def candyCrush(self, board: List[List[int]]) -> List[List[int]]`?